# Write de Melo 2019 as linked ATST

One readout combines Lfar01 at 10^6 and 10^7 PFU/mL, ATCC 27853 at 10^7, and BOIJ02 at 10^7. Each retains infected and matched no-phage triplicates.

In [1]:
from pathlib import Path
import datetime
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import (
    Assay,
    Entities,
    EntityTable,
    FileMetadata,
    Layout,
    Metadata,
    Readings,
    ReadoutIds,
    Study,
)

root = Path.cwd()
source = root / "sources/12866_2019_1481_MOESM7_ESM.xlsx"
workbook = load_workbook(source, data_only=True, read_only=True)
# Each tuple names a source sheet, host, dose, control rows, infected rows, and time column.
groups = {
    "Lfar01_1e6": ("Plan1", "Lfar01", 1000000, 21, 7, 2),
    "Lfar01_1e7": ("Plan1", "Lfar01", 10000000, 51, 37, 2),
    "ATCC27853_1e7": ("Plan2", "ATCC_27853", 10000000, 6, 21, 1),
    "BOIJ02_1e7": ("Plan3", "BOIJ02", 10000000, 7, 22, 1),
}

## Study, assay, and material identities

The paper’s Biological assays section supplies shared conditions. Workbook annotations supply dose labels and replicate columns. Source locations and unresolved details are recorded in `curation.md`.

In [2]:
file_info = FileMetadata(
    data={
        "file_name": "atst_de_melo_2019.atst.txt",
        "format": "ATST",
        "format_version": "0.1",
        "created_on": datetime.date.today().isoformat(),
        "field_delimiter": "TAB",
        "encoding": "UTF-8",
    }
)
study = Study(
    data={
        "title": "Characterization of a bacteriophage with broad host range against strains of Pseudomonas aeruginosa isolated from domestic animals",
        "study_id": "doi:10.1186/s12866-019-1481-z",
        "authors": "Anna Cristhina Carmine de Melo; Amanda da Mata Gomes; Fernando L. Melo; Daniel M. P. Ardisson-Araújo; Agueda Palmira Castagna de Vargas; Valessa Lunkes Ely; Elliot W. Kitajima; Bergmann M. Ribeiro; José Luiz Caldas Wolff",
        "publication_date": "2019",
        "publication_doi": "10.1186/s12866-019-1481-z",
        "description": "BrSP1 infection and matched no-phage controls for three Pseudomonas aeruginosa strains; raw absorbance triplicates from Additional file 8.",
    }
)
entities = Entities(
    tables={
        name: EntityTable(
            name="ENTITIES",
            table_name=name,
            pk=pk,
            data=pd.read_csv(
                root / "entities_definitions" / f"{name}.csv",
                dtype=str,
                keep_default_na=False,
            ),
        )
        for name, pk in [
            ("ISOLATES", "ISOLATE_ID"),
            ("PHAGES", "PHAGE_ID"),
            ("MEDIA", "MEDIUM_ID"),
        ]
    }
)
assay = Assay(
    data={
        "readout_type": "absorbance",
        "readout_unit": "od600",
        "time_unit": "h",
        "wavelength_nm": "600",
        "temperature_c": "36",
        "shaking_speed_rpm": "120",
        "vessel": "10 mL culture in test tubes",
        "medium_id": "TSB_Ca_Mg",
        "medium_supplementation": "CaCl2 40 mg/L; MgSO4 40 mg/L",
        "growth_stage": "early exponential",
        "control_treatment": "SM buffer",
        "duration_h": "24",
        "sampling_times_h": "0, 2, 4, 6, 8, 10, 12, 24",
        "instrument": "NanoDrop 2000c (Thermo Scientific)",
        "replicates_per_condition": "3",
        "source_reference": "Paper p. 11, Biological assays; Fig. 3; Additional file 8",
    }
)

## Raw readings and six-column layouts

Each six-curve source group contributes no-phage and BrSP1 triplicates; the merged readout assigns unique curve IDs across all groups. Mean and standard-deviation cells in Excel are excluded.

In [3]:
def values(sheet, first_row, time_col):
    result = []
    for row in range(first_row, first_row + 8):
        raw_time = sheet.cell(row, time_col).value
        time = float(str(raw_time).replace("T", "").replace("=", "").strip())
        result.append(
            [time, *[sheet.cell(row, time_col + i).value for i in range(1, 4)]]
        )
    return pd.DataFrame(result, columns=["Time", "curve_A", "curve_B", "curve_C"])


readouts = {}
for readout_id, (
    sheet_name,
    host,
    dose,
    control_row,
    infected_row,
    time_col,
) in groups.items():
    sheet = workbook[sheet_name]
    control = values(sheet, control_row, time_col)
    infected = values(sheet, infected_row, time_col)
    assert control["Time"].equals(infected["Time"])
    readings = control.copy()
    readings[["curve_D", "curve_E", "curve_F"]] = infected[
        ["curve_A", "curve_B", "curve_C"]
    ]
    layout = pd.DataFrame(
        [
            {
                "curve_id": f"curve_{letter}",
                "type": "uninfected_control" if i < 3 else "phage_exposed",
                "isolate_id": host,
                "phage_id": "" if i < 3 else "BrSP1",
                "pfu_ml": "0" if i < 3 else str(dose),
                "replicate": str(i % 3 + 1),
            }
            for i, letter in enumerate("ABCDEF")
        ]
    )
    metadata = Metadata(
        data={
            "source_file": "read_shared_data/sources/12866_2019_1481_MOESM7_ESM.xlsx",
            "conversion": "Raw triplicate columns only; workbook mean and SD omitted; time labels normalized to numeric hours",
        }
    )
    readouts[readout_id] = ATSTFile(
        file_info=file_info,
        study=study,
        metadata=metadata,
        assay=assay,
        entities=entities,
        layout=Layout(data=layout),
        readings=Readings(data=readings),
    )
def merged_name(members):
    return "All_strains"

# Combine readouts only when metadata, assay, and recorded times match.
groups_by_context = {}
for readout_id, item in readouts.items():
    key = (tuple(item.metadata.data.items()), tuple(item.assay.data.items()), tuple(item.readings.data["Time"].astype(str)))
    groups_by_context.setdefault(key, []).append((readout_id, item))
combined = {}
for members in groups_by_context.values():
    if len(members) == 1:
        combined[members[0][0]] = members[0][1]
        continue
    layouts = []
    readings = members[0][1].readings.data[["Time"]].copy()
    for _, item in members:
        rename = {}
        for old_id in item.layout.data["curve_id"]:
            letter = get_column_letter(len(readings.columns) + len(rename))
            rename[old_id] = "curve_ID_238" if letter == "ID" else "curve_" + letter
        layout = item.layout.data.copy()
        layout["curve_id"] = layout["curve_id"].map(rename)
        layouts.append(layout)
        readings = pd.concat([readings, item.readings.data.drop(columns="Time").rename(columns=rename).reset_index(drop=True)], axis=1)
    first = members[0][1]
    combined[merged_name(members)] = ATSTFile(file_info=file_info, study=study, metadata=first.metadata, assay=first.assay, entities=entities, layout=Layout(data=pd.concat(layouts, ignore_index=True)), readings=Readings(data=readings))
readouts = combined

readout_ids = ReadoutIds(data=pd.DataFrame({"readout_id": list(readouts)}))
for item in readouts.values():
    item.readout_ids = readout_ids
bundle = MultiReadoutATST(file_info, study, readout_ids, readouts, entities)
validate_atst(bundle)
pd.DataFrame(
    [
        {
            "readout_id": k,
            "curves": len(v.layout.data),
            "time_points": len(v.readings.data),
        }
        for k, v in readouts.items()
    ]
)

In [4]:
output = write_multi_readout_atst(
    bundle, root.parent / "atst_de_melo_2019", linked_files=True
)
print(output)

/Users/juanmantilla/Documents/TKI/ATST/examples/Public data /de Melo 2019/atst_de_melo_2019/atst_de_melo_2019.atst.txt
